In [18]:
import pandas as pd
# read csv
airports = pd.read_csv("../Input/yufei_airports.csv")
flights = pd.read_csv("../Input/flights.csv")
# dimension check
print("airports.csv dimension:", airports.shape)
print("flights_sample_3m.csv dimension:", flights.shape)
# inspecting columns and nulls
print("\nairportscolumns:")
print(airports.info())
print("\nflights columns:")
print(flights.info())
print("\nAirports null counts:")
print(airports.isnull().sum())
print("\nFlights null counts:")
print(flights.isnull().sum())

airports.csv dimension: (322, 7)
flights_sample_3m.csv dimension: (3000000, 32)

airportscolumns:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 322 entries, 0 to 321
Data columns (total 7 columns):
 #   Column     Non-Null Count  Dtype  
---  ------     --------------  -----  
 0   IATA_CODE  322 non-null    object 
 1   AIRPORT    322 non-null    object 
 2   CITY       322 non-null    object 
 3   STATE      322 non-null    object 
 4   COUNTRY    322 non-null    object 
 5   LATITUDE   319 non-null    float64
 6   LONGITUDE  319 non-null    float64
dtypes: float64(2), object(5)
memory usage: 17.7+ KB
None

flights columns:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3000000 entries, 0 to 2999999
Data columns (total 32 columns):
 #   Column                   Dtype  
---  ------                   -----  
 0   FL_DATE                  object 
 1   AIRLINE                  object 
 2   AIRLINE_DOT              object 
 3   AIRLINE_CODE             object 
 4   DOT_CODE        

In [19]:
# cancelled no delayed time
flights_clean = flights[flights['CANCELLED'] == 0].copy()
# diverted don't land in original destination, so arrival delay not accurate
flights_clean = flights_clean[flights_clean['DIVERTED'] == 0]
flights_clean.dropna(subset=['CRS_ELAPSED_TIME'], inplace=True)
# have to keep these so fill with 0 later, these rows may indicates non-delayed or delayed with other reasons, can go back and check after join
delay_cols = ['DELAY_DUE_CARRIER', 'DELAY_DUE_WEATHER', 'DELAY_DUE_NAS', 'DELAY_DUE_SECURITY', 'DELAY_DUE_LATE_AIRCRAFT']
flights_clean[delay_cols] = flights_clean[delay_cols].fillna(0)
# non necessary column
flights_clean.drop(columns=['CANCELLATION_CODE'], inplace=True)
# drop the final 2 rows with missing arrival data
flights_clean.dropna(subset=['ARR_TIME'], inplace=True)
# check dim and null
print(flights_clean.shape)
print(flights_clean.isnull().sum())

(2913802, 31)
FL_DATE                    0
AIRLINE                    0
AIRLINE_DOT                0
AIRLINE_CODE               0
DOT_CODE                   0
FL_NUMBER                  0
ORIGIN                     0
ORIGIN_CITY                0
DEST                       0
DEST_CITY                  0
CRS_DEP_TIME               0
DEP_TIME                   0
DEP_DELAY                  0
TAXI_OUT                   0
WHEELS_OFF                 0
WHEELS_ON                  0
TAXI_IN                    0
CRS_ARR_TIME               0
ARR_TIME                   0
ARR_DELAY                  0
CANCELLED                  0
DIVERTED                   0
CRS_ELAPSED_TIME           0
ELAPSED_TIME               0
AIR_TIME                   0
DISTANCE                   0
DELAY_DUE_CARRIER          0
DELAY_DUE_WEATHER          0
DELAY_DUE_NAS              0
DELAY_DUE_SECURITY         0
DELAY_DUE_LATE_AIRCRAFT    0
dtype: int64


In [25]:
# convert to datetime
flights_clean['FL_DATE'] = pd.to_datetime(flights_clean['FL_DATE'])

def format_time(val):
    val_str = str(int(val)).zfill(4)
    return f"{val_str[:2]}:{val_str[2:]}"

flights_clean['SCHED_DEP_TIMESTAMP'] = pd.to_datetime(
    flights_clean['FL_DATE'].dt.strftime('%Y-%m-%d') + ' ' + flights_clean['CRS_DEP_TIME'].apply(format_time)
)
# round to nearest hour
flights_clean['WEATHER_JOIN_HOUR'] = flights_clean['SCHED_DEP_TIMESTAMP'].dt.round('h')
# change to categorical
categorical_cols = ['AIRLINE', 'AIRLINE_CODE', 'ORIGIN', 'DEST', 'ORIGIN_CITY', 'DEST_CITY']
for col in categorical_cols:
    flights_clean[col] = flights_clean[col].astype('category')
# downcast delay minutes to integers
int_cols = [
    'DEP_DELAY', 'ARR_DELAY', 'TAXI_OUT', 'TAXI_IN', 'AIR_TIME', 'DISTANCE', 'CRS_ELAPSED_TIME', 'ELAPSED_TIME',
    'DELAY_DUE_CARRIER', 'DELAY_DUE_WEATHER', 'DELAY_DUE_NAS', 'DELAY_DUE_SECURITY', 'DELAY_DUE_LATE_AIRCRAFT'
]
for col in int_cols:
    flights_clean[col] = flights_clean[col].astype('int32')
# make sure these columns entry all uppercase
flights_clean['ORIGIN'] = flights_clean['ORIGIN'].str.upper()
flights_clean = flights_clean.drop(columns=["SCHED_DEP_TIMESTAMP"])
flights_clean = flights_clean.rename(columns={
    "ORIGIN": "iata_code",
    "DEST": "dest_iata_code"
})

In [26]:
flights_clean

,FL_DATE,AIRLINE,AIRLINE_DOT,AIRLINE_CODE,DOT_CODE,FL_NUMBER,iata_code,ORIGIN_CITY,dest_iata_code,DEST_CITY,...,CRS_ELAPSED_TIME,ELAPSED_TIME,AIR_TIME,DISTANCE,DELAY_DUE_CARRIER,DELAY_DUE_WEATHER,DELAY_DUE_NAS,DELAY_DUE_SECURITY,DELAY_DUE_LATE_AIRCRAFT,WEATHER_JOIN_HOUR
0,2019-01-09,United Air Lines Inc.,United Air Lines Inc.: UA,UA,19977,1562,FLL,"Fort Lauderdale, FL",EWR,"Newark, NJ",...,186,176,153,1065,0,0,0,0,0,2019-01-09 12:00:00
1,2022-11-19,Delta Air Lines Inc.,Delta Air Lines Inc.: DL,DL,19790,1149,MSP,"Minneapolis, MN",SEA,"Seattle, WA",...,235,236,189,1399,0,0,0,0,0,2022-11-19 21:00:00
2,2022-07-22,United Air Lines Inc.,United Air Lines Inc.: UA,UA,19977,459,DEN,"Denver, CO",MSP,"Minneapolis, MN",...,118,112,87,680,0,0,0,0,0,2022-07-22 10:00:00
3,2023-03-06,Delta Air Lines Inc.,Delta Air Lines Inc.: DL,DL,19790,2295,MSP,"Minneapolis, MN",SFO,"San Francisco, CA",...,260,285,249,1589,0,0,24,0,0,2023-03-06 16:00:00
4,2020-02-23,Spirit Air Lines,Spirit Air Lines: NK,NK,20416,407,MCO,"Orlando, FL",DFW,"Dallas/Fort Worth, TX",...,181,182,153,985,0,0,0,0,0,2020-02-23 19:00:00
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2999995,2022-11-13,American Airlines Inc.,American Airlines Inc.: AA,AA,19805,1522,JAX,"Jacksonville, FL",CLT,"Charlotte, NC",...,85,71,55,328,0,0,0,0,0,2022-11-13 18:00:00
2999996,2022-11-02,American Airlines Inc.,American Airlines Inc.: AA,AA,19805,1535,ORD,"Chicago, IL",AUS,"Austin, TX",...,176,145,130,977,0,0,0,0,0,2022-11-02 13:00:00
2999997,2022-09-11,Delta Air Lines Inc.,Delta Air Lines Inc.: DL,DL,19790,2745,HSV,"Huntsville, AL",ATL,"Atlanta, GA",...,55,50,28,151,0,36,0,0,0,2022-09-11 06:00:00
2999998,2019-11-13,Republic Airline,Republic Airline: YX,YX,20452,6134,BOS,"Boston, MA",LGA,"New York, NY",...,88,77,50,184,0,0,0,0,0,2019-11-13 16:00:00


In [27]:
print(flights_clean.shape)
print(flights_clean.info())
print(flights_clean.isnull().sum())

(2913802, 32)
<class 'pandas.core.frame.DataFrame'>
Index: 2913802 entries, 0 to 2999999
Data columns (total 32 columns):
 #   Column                   Dtype         
---  ------                   -----         
 0   FL_DATE                  datetime64[ns]
 1   AIRLINE                  category      
 2   AIRLINE_DOT              object        
 3   AIRLINE_CODE             category      
 4   DOT_CODE                 int64         
 5   FL_NUMBER                int64         
 6   iata_code                object        
 7   ORIGIN_CITY              category      
 8   dest_iata_code           category      
 9   DEST_CITY                category      
 10  CRS_DEP_TIME             int64         
 11  DEP_TIME                 float64       
 12  DEP_DELAY                int32         
 13  TAXI_OUT                 int32         
 14  WHEELS_OFF               float64       
 15  WHEELS_ON                float64       
 16  TAXI_IN                  int32         
 17  CRS_ARR_TIME      

In [28]:
flights_clean.to_csv("../Output/yufei_flights_cleaned.csv", index=False)